# Resume Corrected Static 3DSSG Training

This notebook resumes the interrupted corrected training from the newest completed checkpoint and continues to epoch 50. It does not regenerate RGB, LiDAR, text, or the static database.

Run it in a fresh kernel. Do not run the original training cell at the same time.


In [ ]:
from pathlib import Path
import re

ROOT = Path(r"D:\MTP_Project\MTP_Pipeline_3RScan")
RUN_ROOT = ROOT / "pipeline_outputs" / "official_static_pretrained_pointnet_run"
CHECKPOINT_DIR = RUN_ROOT / "checkpoints"
DATABASE = ROOT / "pipeline_outputs" / "3rscan_official_static_database_v2"
OFFICIAL_SPLITS = ROOT / "official_splits"

completed = list(CHECKPOINT_DIR.glob("integration_by_parts_3rscan_epoch_*.pt"))
assert completed, "No completed checkpoint was found."
resume_checkpoint = max(completed, key=lambda path: int(re.search(r"epoch_(\d+)$", path.stem).group(1)))
print(f"Resuming from: {resume_checkpoint.name}")


In [ ]:
import argparse
from mtp_pipeline.config import ProjectPaths
from mtp_pipeline.train import train

args = argparse.Namespace(
    reference_root=ProjectPaths().reference_root,
    output_root=RUN_ROOT,
    database=DATABASE,
    mode="static",
    train_scans=OFFICIAL_SPLITS / "train_scans.txt",
    val_scans=OFFICIAL_SPLITS / "validation_scans.txt",
    split_seed=42,
    seed=42,
    # Total target, not additional epochs. The resume code continues from the checkpoint epoch.
    epochs=50,
    max_scenes=None,
    max_frames=None,
    learning_rate=1e-4,
    resume_checkpoint=resume_checkpoint,
    negative_ratio=None,
    lambda_temporal=0.0,
    lambda_temporal_part=0.0,
    lambda_node=1.0,
    lambda_edge=1.0,
    lambda_dynamic=0.0,
    grad_clip=1.0,
    device="cuda",
    shuffle=True,
    num_parts=7,
    ablation="none",
)

checkpoint_path = train(args)
print(f"Training finished. Final checkpoint: {checkpoint_path}")
